# Retrieval Augmented Generation (RAG)

$\bullet$ Retrieval-Augmented Generation (RAG) is a framework that enhances large language models (LLMs) by combining a non-generative retrieval mechanism with a generative model, enabling the system to produce responses grounded in externally retrieved information.  
$\bullet$ Here are two behaviours that are often observed as problematic when interacting with large language models.  
$\qquad \blacksquare$ No source  
$\qquad \blacksquare$ Out of date  
$\bullet$ RAG approach improves factual accuracy and reduces hallucinations by grounding the generated responses in real, up-to-date information.  
$\bullet$ Typically, a RAG system consists of two main components:  
$\qquad \blacksquare$ Retriever  
$\qquad \blacksquare$ Generator  
$\bullet$ Formally, let $x$ be the input query and $D = \{d_1, d_2, \dots, d_n\}$ be a collection of documents. The retrieval component selects a subset of relevant documents $D_k \subset D$ based on a similarity function:

$$
D_k = \text{Retriever}(x, D)
$$

$\bullet$ The generator then produces an output $y$ conditioned on both the input query and the retrieved documents:

$$
y \sim P(y \mid x, D_k)
$$

$\bullet$ The retriever uses the query to fetch relevant documents first, and only then the LLM generates the answer using query and those documents.  
$\bullet$ The overall pipeline can be summarized as:  

$$
x \rightarrow \text{Retriever} \rightarrow (x,D_k) \rightarrow \text{Generator} \rightarrow y
$$

$\bullet$ Advantages of RAG:  
$\qquad \star$ Access to external and up-to-date knowledge.  
$\qquad \star$ Reduced need for frequent model retraining.  
$\qquad \star$ Improved interpretability through retrieved evidence.  
$\bullet$ Limitations of RAG:  
$\qquad \star$ Performance depends on retrieval quality.  
$\qquad \star$ Increased system complexity and latency.

$\bullet$ Why can't we fine tune an LLM model with our private data (Ex: Company policies, HR data, Finance details)?  
$\qquad \blacksquare$ It takes a higher computational cost (needs to fine tune billions of parameters)  
$\qquad \blacksquare$ Need to fine tune the model continuously (as data getting updated all the time)

$\bullet$ Data Ingestion Pipeline - The way of storing all the external documents and data as vectors in a vector storage
$$
\textrm{Data(PDF, HTML, Excel, SQL DBs)} \rightarrow \textrm{Parsing} \rightarrow \textrm{Chunking} \rightarrow \textrm{Embeddings(Text→Vectors)} \rightarrow \textrm{Vector DB(Vector Storage)}
$$

$\bullet$ Retrieval Pipeline of a Traditional RAG - The way of a query becomes a vector and then finds similar data inside the vector storage using similarity search functions and then going to the LLM with a prompt
$$
\textrm{Query} \rightarrow \textrm{Embeddings(Text→Vectors)} \rightarrow \textrm{VectorDB(Vector Storage)} \rightarrow \textrm{Context} \rightarrow \textrm{Augmentation(Context + Prompt)} \rightarrow \textrm{LLM}
$$

In [2]:
# importing essential libraries
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

### Document Structure
$\bullet$ Document structure contain two main parts,  
$\qquad \blacksquare$ Page Content  
$\qquad \blacksquare$ Meta Data

Here, we are loading one text file as a Langchain Document object

In [3]:
loader = TextLoader("Data/TextFiles/introduction.txt")
document = loader.load()
document

[Document(metadata={'source': 'Data/TextFiles/introduction.txt'}, page_content='Dilan Senuruk the first son of Vortex the Great in kingdom of Aerathis has invented the first chain reaction simulator of the sun. Dilan Senuruk himself is known as the Great sun god Nika which is a name given to the best scientific creation. He was married to the beauty queen Maleesha Ruvindi of Galanthrea and he is also a father to 3 children.\n\nThe First chain reaction simulator of the sun was made in 1967 and made public in 1971 so that every scientist in the continent can use it for research purposes. The chain reaction simulator works in fore steps and they are,\n1. The starting phase - Dummy photons and electrons are fed into the main valve\n2. Radiation phase - The top window opens for the sun radiation to fall on the main core\n3. Generating phase - The photos and electrons will combine to create bronton with the sun radiation\n4. Graphing phase - The bronton will fall on to the glass illuminating

Here, we are loading a whole directory with four PDFs as Langchain Document objects

In [4]:
dir_loader = DirectoryLoader("Data/PDFs", glob="**/*.pdf", loader_cls=PyMuPDFLoader, show_progress=False)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T10:48:43+05:30', 'source': 'Data\\PDFs\\Findings.pdf', 'file_path': 'Data\\PDFs\\Findings.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Dilan Senuruk', 'subject': '', 'keywords': '', 'moddate': '2026-04-17T10:48:43+05:30', 'trapped': '', 'modDate': "D:20260417104843+05'30'", 'creationDate': "D:20260417104843+05'30'", 'page': 0}, page_content='Findings of Vortex the Great \nKing Vortex the Great’s obsession with science began as a quiet curiosity but evolved into a \nrelentless pursuit that reshaped the intellectual foundations of his kingdom. Unlike traditional \nrulers who relied on inherited knowledge, Vortex established vast observatories and laboratories \nsuspended between the floating isles of Aerathis, where he could study the behavior of storms at \ntheir source. His earliest breakthrough came with the formulation of what he called Aero-\

---
$$
\textrm{\Large{Data Ingestion Pipeline}}
$$

### Parsing / Loading

In [5]:
# Read all the PDFs inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"    ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"    ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs("data")


found 4 PDF files to process

Processing: Findings.pdf
    ✓ Loaded 1 pages

Processing: Intro.pdf
    ✓ Loaded 1 pages

Processing: Kingdom_Expansion.pdf
    ✓ Loaded 1 pages

Processing: Sons.pdf
    ✓ Loaded 2 pages

Total documents loaded: 5


### Chunking / Splitting

Chunking is the process of dividing large documents into smaller, manageable pieces to improve retrieval efficiency and semantic relevance. Given a document $ D $, it is split into a set of smaller segments:

$$
D \rightarrow \{c_1, c_2, c_3, \dots, c_n\}
$$

where each $ c_i $ represents a chunk of text.

Chunking is necessary because large language models and embedding models have input size limitations, and smaller chunks allow for more precise semantic matching during retrieval. Often, overlapping chunks are used to preserve context:

$$
c_i = \text{tokens}[i \cdot s : i \cdot s + k] \qquad \qquad i = \{0, 1, 2, 3, ... , n\}
$$

where:
- $ k $ = chunk size  
- $ s $ = stride (overlap control)


In [6]:
# Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show exxample of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunks = split_documents(all_pdf_documents)

Split 5 documents into 16 chunks

Example chunk:
Content: Findings of Vortex the Great 
King Vortex the Great’s obsession with science began as a quiet curiosity but evolved into a 
relentless pursuit that reshaped the intellectual foundations of his kingdom...
Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T10:48:43+05:30', 'author': 'Dilan Senuruk', 'moddate': '2026-04-17T10:48:43+05:30', 'source': 'data\\PDFs\\Findings.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Findings.pdf', 'file_type': 'pdf'}


### Generating Embeddings

Each chunk is transformed into a dense vector representation using an embedding model. This process maps text into a high-dimensional semantic space:

$$
c_i \xrightarrow{\text{Embedding Model}} \mathbf{v}_i \in \mathbb{R}^d
$$

where:
- $ \mathbf{v}_i $ is the embedding vector of chunk $ c_i $  
- $ d $ is the embedding dimension (e.g., 384 for MiniLM)

The embedding captures the semantic meaning of the text, allowing similar chunks to be located based on vector similarity rather than keyword matching.

In [7]:
# Handles document embedding generation using SentenceTransformer
# all-MiniLM-L6-v2 <-- HuggingFace model name for sentence embeddings
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    # Load the SentenceTransformer model
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    # Generate embeddings for a list of texts
    # Args: texts <-- List of text strings to embed
    # Returns: numpy array of embeddings with shape (len(texts), embedding_dim)
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9454.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [8]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 16 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

Generated embeddings with shape: (16, 384)


### Storing in Vector DataBase

The generated embeddings are stored in a vector database for efficient similarity search:

$$
\mathcal{V} = \{(\mathbf{v}_i, c_i, m_i)\}_{i=1}^{n}
$$

where:
- $ \mathbf{v}_i $ = embedding vector  
- $ c_i $ = original text chunk  
- $ m_i $ = metadata (e.g., source document, page number)

In [9]:
# Manages document embeddings in a ChromaDB vector store
# Args: collection_name <-- Name of the ChromaDB collection
# Args: persist_directory <-- Directory to persist the vector store
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./Data/VectorStore"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    # Initialize ChromaDB client and collection
    def _initialize_store(self):
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document embeddings for RAG"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    # Add documents and their embeddings to the vector store
    # Args: documents <-- List of Langchain documents
    # Args: embeddings <-- Corresponding embeddings for the documents
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documentS to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(ids=ids, embeddings=embeddings_list, metadatas=metadatas, documents=documents_text)
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
vectorstore.add_documents(chunks, embeddings)

Adding 16 documentS to vector store...
Successfully added 16 documents to vector store
Total documents in collection: 16


---
$$
\textrm{\Large{Query Retrieval Pipeline}}
$$

When a user submits a query $ q $, it is also embedded:

$$
q \xrightarrow{\text{Embedding Model}} \mathbf{v}_q
$$

The system retrieves the most relevant chunks by computing similarity (e.g., cosine similarity):

$$
\text{sim}(\mathbf{v}_q, \mathbf{v}_i) = \frac{\mathbf{v}_q \cdot \mathbf{v}_i}{\|\mathbf{v}_q\| \|\mathbf{v}_i\|}
$$

The top-$ k $ most similar chunks are then passed to the language model for response generation.

In [11]:
# Handles query-based retrieval from the vector store
# Args: vector_store <-- Vector store containing document embeddings
# Args: embedding_manager <-- Manager for generating query embeddings
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    # Retrieve relevant documents for a query
    # Args: top_k <-- Number of top results to return
    # Args: score_threshold <-- Minimum similarity score threshold
    # Returns: List of dictionaries containing retrieved documents and metadata
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # Search in vector store
        try:
            results = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({'id': doc_id, 'content': document, 'metadata': metadata, 'similarity_score': similarity_score, 'distance': distance, 'rank': i+1})
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [12]:
rag_retriever.retrieve("How did king Vortex's first son Dilan Senuruk died?")

Retrieving documents for query: 'How did king Vortex's first son Dilan Senuruk died?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_ef9f0483_11',
  'content': 'Sons of Vortex the Great \nAfter the assassination of Dilan Senuruk—the brilliant heir known as the “Great Sun God \nNika”—King Vortex abandoned the capital of Aerathis and relocated his seat of power to the \nremote and heavily fortified kingdom of Astra Nocturne, a land perpetually covered in twilight \nskies. There, he became more withdrawn, colder, and far more authoritarian. It was in this later \nperiod of his life that the records speak of his seven sons, each raised under vastly different \nconditions than Dilan, shaped by loss, fear, and the shadow of their father’s wrath. \nThe Seven Sons of King Vortex \n1. Aethon Vortex – The Heir of Silence \nThe eldest among the seven, Aethon was chosen as the new successor after Dilan’s death. Unlike \nhis father, he rejected emotional extremes and trained himself to suppress all feeling. He \nspecialized in atmospheric compression techniques, capable of creating zones where sound itself',
  'meta

### A Simple Implementation of Query Retrieval and LLM Calling

In [13]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)
# Simple rag function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    # Generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely. Context: {context}, Question: {query}, Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [14]:
answer = rag_simple("How did king Vortex's first son Dilan Senuruk died?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'How did king Vortex's first son Dilan Senuruk died?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 107.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


The text does not explicitly state how Dilan Senuruk (also known as the "Great Sun God Nika") died. It only mentions that he was assassinated.


### An Advanced Implementation of Query Retrieval and LLM Calling

In [16]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    # RAG pipeline with extra features: Returns answer, sources, confidence score, and optionally full context.
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{'sources': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')), 'page': doc['metadata'].get('page', 'unknown'),
                'source': doc['similarity_score'], 'preview': doc['content'][:300] + '...'} for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    output = {'answer': response.content, 'sources': sources, 'confidence': confidence}
    if return_context:
        output['context'] = context
    return output

# Example Usage:
result = rag_advanced("How did king Vortex's first son Dilan Senuruk died?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'How did king Vortex's first son Dilan Senuruk died?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: Dilan Senuruk, also known as the "Great Sun God Nika," was assassinated.
Sources: [{'sources': 'Sons.pdf', 'page': 0, 'source': 0.3590034246444702, 'preview': 'Sons of Vortex the Great \nAfter the assassination of Dilan Senuruk—the brilliant heir known as the “Great Sun God \nNika”—King Vortex abandoned the capital of Aerathis and relocated his seat of power to the \nremote and heavily fortified kingdom of Astra Nocturne, a land perpetually covered in twiligh...'}]
Confidence: 0.3590034246444702
Context Preview: Sons of Vortex the Great 
After the assassination of Dilan Senuruk—the brilliant heir known as the “Great Sun God 
Nika”—King Vortex abandoned the capital of Aerathis and relocated his seat of power to the 
remote and heavily fortified kingdom of Astra Nocturne, a land perpetually covered in twiligh
